# 02 — Pré-processamento

**Dimensão 4 da rúbrica — 15 pontos.**

Cada decisão precisa de justificativa escrita. Decidir *não* criar features é
aceitável, desde que o motivo esteja explícito.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np
# Semente fixa: use em TODO ponto com aleatoriedade (split, modelos, CV).
RANDOM_STATE = 42

RAW_CREDIT_RECORD= Path("..") / "data" / "raw" / "credit_record.csv"          # PREENCHER: nome do arquivo
RAW_APPLICATION_RECORD= Path("..") / "data" / "raw" / "application_record.csv"     # PREENCHER: nome do arquivo
PROCESSED = Path("..") / "data" / "processed"
TARGET = "target"                                            # PREENCHER: variável alvo

pd.set_option("display.max_columns", None)

/home/markosokada/.local/lib/python3.12/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.9.0' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/markosokada/.local/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
# Criando cópias para não modificar o DATASET principal
df_credit_record = pd.read_csv(RAW_CREDIT_RECORD)
df_application_record = pd.read_csv(RAW_APPLICATION_RECORD)

## 1. Dados faltantes

### Tratamento de IDs duplicados em `application_record`
**Justificativa:** Foi identificada a presença de 47 IDs duplicados na base `application_record` contendo informações cadastrais conflitantes.  

Como o `ID`  serve como chave de união com a variável alvo, manter essas duplicatas geraria  inconsistências. Removeu-se todas as linhas associadas a esses IDs conflitantes para preservar a unicidade do cadastro.

In [3]:
print("Formato da base antes tratamento:", df_application_record.shape)
# Identificar IDs duplicados
ids_duplicados_df_application_recard= df_application_record.loc[df_application_record["ID"].duplicated(keep=False), "ID"].unique()


# Remover todos os registros com IDs duplicados
df_application_record = df_application_record[~df_application_record["ID"].isin(ids_duplicados_df_application_recard)].copy()
print("Formato da base após tratamento:", df_application_record.shape)

Formato da base antes tratamento: (438557, 18)
Formato da base após tratamento: (438463, 18)


### Tratamento de valores ausentes em `OCCUPATION_TYPE`

**Justificativa/Decisão:** A coluna possui uma quantidade expressiva de dados ausentes (134.177 linhas), em sua grande maioria associados a aposentados (`Pensioner`). Como excluir essas linhas causaria perda severa de dados válidos, optou-se por preenchê-los com a categoria textual `"Unknown"`, transformando a ausência em uma informação categórica útil para o modelo.

In [4]:
print("quantidades de nulos antes do tratamento: ", df_application_record["OCCUPATION_TYPE"].isna().sum())


df_application_record["OCCUPATION_TYPE"] = df_application_record["OCCUPATION_TYPE"].fillna("Unknown")

print("quantidades de nulos pós do tratamento: ", df_application_record["OCCUPATION_TYPE"].isna().sum())

quantidades de nulos antes do tratamento:  134177
quantidades de nulos pós do tratamento:  0


## 2. Definição da variável alvo
_Se houve binarização, qual limiar e por quê? Justifique com base na distribuição, não por convenção._

**Justificativa da binarização:**  
A variável original `STATUS` possui 8 categorias ordinais/textuais. Para transformá-la em classificação binária, o limiar escolhido foi de atrasos $\ge$ 30 dias (`STATUS` 1, 2, 3, 4 e 5) definidos como **Mau Pagador (1)**, conforme os padrões de mercado (inadimplência grave). Os meses com atrasos leves, quitados ou sem empréstimos foram mapeados como **Bom Pagador (0)**. No nível do cliente, considerou-se o pior caso histórico (máximo). A distribuição final mostra forte desbalanceamento (1,69% de maus pagadores na base fundida).


In [5]:
# Criar indicador de mês com atraso grave
df_credit_record["BAD_MONTH"] = df_credit_record["STATUS"].isin(["1","2","3", "4", "5"]).astype(int)

# Consolidar a variável alvo por cliente (ID)
target = df_credit_record.groupby("ID")["BAD_MONTH"].max().reset_index()
target = target.rename(columns={"BAD_MONTH": "TARGET_BAD"})


### Junção das bases `application_record` e `target`

**Justificativa:** Nem todos os clientes cadastrados possuem histórico de crédito e vice-versa. Utilizou-se um `merge` do tipo `inner` (interseção) para garantir que a base final contenha estritamente clientes que possuem atributos preditivos e rótulo de destino associado, resultando em 36.457 registros únicos.


In [6]:
df = df_application_record.merge(target, on="ID", how="inner")# Fazendo INNER JOIN 

In [7]:
df.shape# verificando o tamanho do dataset após o Inner Join

(36457, 19)

## 3. Normalização / padronização

**Escolha e justificativa:** *Nenhum neste momento.* Conforme instruído pelo 
template de arquitetura, técnicas como `StandardScaler` ou `MinMaxScaler` devem 
ser aplicadas estritamente dentro de um `Pipeline` de Machine Learning na etapa 
de treinamento (Notebook 03). Executar o ajuste do scaler antes do split de 
treino/teste causaria vazamento de informação (*data leakage*).

## 4. Feature engineering

**Justificativa:** _preencher._

### Justificativa das novas variáveis derivadas:'
1. `AGE_YEARS`: Criada a partir de `DAYS_BIRTH` para converter dias negativos em 
uma escala numérica positiva e altamente interpretável (Idade em Anos).
2. `EMPLOYED_SPECIAL_VALUE` e `EMPLOYED_YEARS`: A coluna original `DAYS_EMPLOYED` 
continha o valor anômalo `365243` para designar clientes que não trabalham 
atualmente (ex.: aposentados). Esse valor técnico enviesaria modelos lineares 
ou de distância. Criou-se uma flag binária para capturar essa condição especial 
e preencheu-se o tempo em anos com `-1` para isolar semanticamente esses casos.
3. `LOG_INCOME`: A variável de renda (`AMT_INCOME_TOTAL`) exibe forte assimetria 
à direita. A transformação por Logaritmo (`log1p`) estabiliza a variância e 
reduz o impacto de outliers de alta renda sem descartar dados.
4. **Remoção de Redundâncias e Variável Constante:** As colunas originais em dias 
foram descartadas para evitar colinearidade. A coluna `FLAG_MOBIL` foi removida 
pois continha valor único `1` em todos os registros, não possuindo variância 
ou poder discriminatório.

In [8]:
# 1. Idade interpretável em anos
df["AGE_YEARS"] = (-df["DAYS_BIRTH"] / 365).round(1)

# 2. Tratamento do valor anômalo de emprego
df["EMPLOYED_SPECIAL_VALUE"] = (df["DAYS_EMPLOYED"] == 365243).astype(int)
df["DAYS_EMPLOYED_TREATED"] = df["DAYS_EMPLOYED"].replace(365243, pd.NA)
df["EMPLOYED_YEARS"] = (-pd.to_numeric(df["DAYS_EMPLOYED_TREATED"], errors="coerce") / 365).round(1)
df["EMPLOYED_YEARS"] = df["EMPLOYED_YEARS"].fillna(-1)

# 3. Transformação logarítmica da renda para suavizar assimetria
df["LOG_INCOME"] = np.log1p(df["AMT_INCOME_TOTAL"])

# 4. Mapeamento de variáveis categóricas binárias Y/N para 1/0
df["FLAG_OWN_CAR"] = np.where(df["FLAG_OWN_CAR"].isin(["Y", 1, "1"]), 1, 0)
df["FLAG_OWN_REALTY"] = np.where(df["FLAG_OWN_REALTY"].isin(["Y", 1, "1"]), 1, 0)
df["CODE_GENDER"] = np.where(df["CODE_GENDER"].isin(["M", 1, "1"]), 1, 0)

# Remoção de colunas originais redundantes e colunas sem variância (FLAG_MOBIL)
df_application_record_tratado = df.drop(
    columns=[
        "DAYS_BIRTH",
        "DAYS_EMPLOYED",
        "DAYS_EMPLOYED_TREATED",
        "FLAG_MOBIL"
    ]
)

# 

## 5. Salvar dataset tratado

In [9]:
PROCESSED.mkdir(parents=True, exist_ok=True)
df_application_record_tratado .to_csv(PROCESSED / "df_application_record_tratado.csv", index=False)

print(f"✅ Pipeline concluído. Dataset salvo em: {PROCESSED / 'df_application_record_tratado.csv'}")
print(f"Formato final dos dados: {df_application_record_tratado.shape}")


✅ Pipeline concluído. Dataset salvo em: ../data/processed/df_application_record_tratado.csv
Formato final dos dados: (36457, 20)
